# Rich Pre-fight Features: RF And XGBoost

This notebook builds a richer pre-fight feature set from `data_clean/all_fights.csv`, using ideas from the older notebooks:

- streaks, wins/losses/draws
- total rounds fought and title bouts
- historical win-by counts
- height, reach, weight, age, stance
- weight class, title bout, scheduled rounds
- pre-fight career stat differences
- Elo and `DIFF_elo`
- sign-aware `DIFF_age_centered_sq`

Rows with `Sideways` or `Open Stance` are removed to avoid rare stance artifacts. Validation uses 5 random 90/10 stratified splits for Random Forest and XGBoost.

In [2]:
from collections import defaultdict
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
SPLIT_SEEDS = [11, 22, 33, 44, 55]
TEST_SIZE = 0.10

ELO_START = 1500.0
ELO_K = 32.0

RARE_STANCES_TO_DROP = {"Sideways", "Open Stance", "OpenStance", "Open"}

ALL_FIGHTS_PATH = Path("..") / "data_clean" / "all_fights.csv"
OPTIONAL_CONTEXT_PATH = Path("..") / "old" / "ufc-master.csv"

RF_PARAMS = dict(
    n_estimators=600,
    min_samples_leaf=8,
    max_features="sqrt",
    n_jobs=-1,
    class_weight=None,
)

XGB_PARAMS = dict(
    n_estimators=700,
    max_depth=3,
    learning_rate=0.03,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=2.0,
    objective="binary:logistic",
    eval_metric="logloss",
    n_jobs=-1,
)

## Load, Parse, And Add Context

In [3]:
def parse_height_inches(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    match = re.search(r"(\d+)\s*'\s*(\d+)", text)
    if match:
        return int(match.group(1)) * 12 + int(match.group(2))
    return pd.to_numeric(text, errors="coerce")

def parse_reach_inches(value):
    if pd.isna(value):
        return np.nan
    return pd.to_numeric(str(value).replace('"', '').strip(), errors="coerce")

def parse_clock_seconds(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if not text or text == "--":
        return np.nan
    parts = text.split(":")
    if len(parts) != 2:
        return np.nan
    return int(parts[0]) * 60 + int(parts[1])

def parse_round_lengths(format_text):
    if pd.isna(format_text):
        return []
    text = str(format_text)
    match = re.search(r"\(([^)]*)\)", text)
    if not match:
        return []
    lengths = []
    for part in match.group(1).split("-"):
        part = part.strip()
        if part.isdigit():
            lengths.append(int(part) * 60)
    return lengths

def parse_no_of_rounds(format_text):
    lengths = parse_round_lengths(format_text)
    if lengths:
        return len(lengths)
    match = re.search(r"(\d+)\s*Rnd", str(format_text))
    return int(match.group(1)) if match else np.nan

def total_fight_seconds(row):
    last_round = pd.to_numeric(row.get("last_round"), errors="coerce")
    last_time = parse_clock_seconds(row.get("last_round_time"))
    lengths = parse_round_lengths(row.get("format"))
    if pd.isna(last_round) or pd.isna(last_time):
        return np.nan
    last_round = int(last_round)
    if lengths:
        prior = sum(lengths[:max(last_round - 1, 0)])
    else:
        prior = max(last_round - 1, 0) * 5 * 60
    return prior + last_time

def clean_stance(value):
    if pd.isna(value):
        return "Unknown"
    text = str(value).strip()
    if text == "":
        return "Unknown"
    return text

def normalize_bool(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    return str(value).strip().lower() in {"true", "1", "yes"}

df = pd.read_csv(ALL_FIGHTS_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["winner_side"] = np.select(
    [df["winner"].eq(df["R_fighter"]), df["winner"].eq(df["B_fighter"])],
    ["R", "B"],
    default="",
)
df["winner_side"] = df["winner_side"].replace("", np.nan)
df = df.dropna(subset=["date", "winner_side", "R_fighter", "B_fighter"]).copy()

if OPTIONAL_CONTEXT_PATH.exists():
    context_cols = [
        "date", "R_fighter", "B_fighter", "title_bout", "weight_class", "no_of_rounds",
        "R_Weight_lbs", "B_Weight_lbs",
    ]
    context = pd.read_csv(OPTIONAL_CONTEXT_PATH, usecols=lambda c: c in context_cols)
    context["date"] = pd.to_datetime(context["date"], errors="coerce")
    context = context.drop_duplicates(["date", "R_fighter", "B_fighter"])
    context = context.rename(columns={
        "title_bout": "ctx_title_bout",
        "weight_class": "ctx_weight_class",
        "no_of_rounds": "ctx_no_of_rounds",
        "R_Weight_lbs": "ctx_R_Weight_lbs",
        "B_Weight_lbs": "ctx_B_Weight_lbs",
    })
    df = df.merge(context, on=["date", "R_fighter", "B_fighter"], how="left")
else:
    print("Optional context file not found; weight, weight_class, title_bout may be missing.")

df["title_bout"] = df.get("ctx_title_bout", False).map(normalize_bool) if "ctx_title_bout" in df else False
df["weight_class"] = df.get("ctx_weight_class", "Unknown").fillna("Unknown") if "ctx_weight_class" in df else "Unknown"
df["no_of_rounds"] = pd.to_numeric(df.get("ctx_no_of_rounds"), errors="coerce") if "ctx_no_of_rounds" in df else np.nan
df["no_of_rounds"] = df["no_of_rounds"].fillna(df["format"].map(parse_no_of_rounds))

for side in ["R", "B"]:
    df[f"{side}_DOB_dt"] = pd.to_datetime(df[f"{side}_DOB"], errors="coerce")
    df[f"{side}_age"] = (df["date"] - df[f"{side}_DOB_dt"]).dt.days / 365.25
    df[f"{side}_Height_inches"] = df[f"{side}_Height"].map(parse_height_inches)
    df[f"{side}_Reach_inches"] = df[f"{side}_Reach"].map(parse_reach_inches)
    df[f"{side}_Weight_lbs"] = pd.to_numeric(df.get(f"ctx_{side}_Weight_lbs"), errors="coerce") if f"ctx_{side}_Weight_lbs" in df else np.nan
    df[f"{side}_Stance_clean"] = df[f"{side}_Stance"].map(clean_stance)
    df[f"{side}_CTRL_seconds"] = df[f"{side}_CTRL"].map(parse_clock_seconds)

df["fight_time_seconds"] = df.apply(total_fight_seconds, axis=1)
df["last_round"] = pd.to_numeric(df["last_round"], errors="coerce")
df = df.sort_values(["date", "R_fighter", "B_fighter"]).reset_index(drop=True)
df["fight_id"] = np.arange(len(df))

before_stance_drop = len(df)
df = df[
    ~df["R_Stance_clean"].isin(RARE_STANCES_TO_DROP)
    & ~df["B_Stance_clean"].isin(RARE_STANCES_TO_DROP)
].copy().reset_index(drop=True)
df["fight_id"] = np.arange(len(df))

print("Loaded:", ALL_FIGHTS_PATH)
print("Rows after valid winner filter:", before_stance_drop)
print("Rows after dropping rare stance rows:", len(df))
print("Dropped rare stance rows:", before_stance_drop - len(df))
print("Context matched title_bout rows:", int(df["title_bout"].notna().sum()))
print("Weight non-null rows:", int(df[["R_Weight_lbs", "B_Weight_lbs"]].notna().all(axis=1).sum()))
print(df["winner_side"].value_counts())
print("Majority baseline:", df["winner_side"].value_counts(normalize=True).max().round(4))

Loaded: ..\data_clean\all_fights.csv
Rows after valid winner filter: 8549
Rows after dropping rare stance rows: 8520
Dropped rare stance rows: 29
Context matched title_bout rows: 8520
Weight non-null rows: 6793
winner_side
R    5452
B    3068
Name: count, dtype: int64
Majority baseline: 0.6399


## Build Pre-fight Career Stats And Elo

In [4]:
STAT_GROUPS = ["SIG_STR", "TOTAL_STR", "TD", "HEAD", "BODY", "LEG", "DISTANCE", "CLINCH", "GROUND"]
COUNT_STATS = ["KD", "SUB_ATT", "REV"]

for side in ["R", "B"]:
    for col in [f"{side}_{s}" for s in COUNT_STATS] + [f"{side}_{g}_{x}" for g in STAT_GROUPS for x in ["landed", "att"]] + [f"{side}_CTRL_seconds"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

def simplify_win_by(value):
    text = "" if pd.isna(value) else str(value).lower()
    if "submission" in text:
        return "SUB"
    if "ko" in text or "tko" in text:
        return "KO_TKO"
    if "decision" in text:
        return "DEC"
    return "OTHER"

def new_state():
    return {
        "total_fights": 0,
        "wins": 0,
        "losses": 0,
        "draw": 0,
        "current_win_streak": 0,
        "current_lose_streak": 0,
        "longest_win_streak": 0,
        "total_rounds_fought": 0.0,
        "total_title_bouts": 0.0,
        "total_fight_time_seconds": 0.0,
        "win_by_DEC": 0.0,
        "win_by_KO_TKO": 0.0,
        "win_by_SUB": 0.0,
        "win_by_OTHER": 0.0,
        "sums": defaultdict(float),
    }

def rate(num, den):
    return np.nan if den == 0 or pd.isna(den) else num / den

def per_min(num, seconds):
    return np.nan if seconds == 0 or pd.isna(seconds) else num / (seconds / 60.0)

def state_to_features(state):
    s = state["sums"]
    seconds = state["total_fight_time_seconds"]
    feats = {
        "total_fights": state["total_fights"],
        "wins": state["wins"],
        "losses": state["losses"],
        "draw": state["draw"],
        "win_pct": rate(state["wins"], state["total_fights"]),
        "current_win_streak": state["current_win_streak"],
        "current_lose_streak": state["current_lose_streak"],
        "longest_win_streak": state["longest_win_streak"],
        "total_rounds_fought": state["total_rounds_fought"],
        "total_title_bouts": state["total_title_bouts"],
        "total_fight_time_seconds": seconds,
        "fight_quickness": rate(state["total_fights"], seconds / 60.0),
        "win_by_DEC": state["win_by_DEC"],
        "win_by_KO_TKO": state["win_by_KO_TKO"],
        "win_by_SUB": state["win_by_SUB"],
        "win_by_OTHER": state["win_by_OTHER"],
    }
    for count_stat in COUNT_STATS:
        feats[f"{count_stat}_per_min"] = per_min(s[count_stat], seconds)
    feats["CTRL_pct"] = rate(s["CTRL_seconds"], seconds)
    for group in STAT_GROUPS:
        landed = s[f"{group}_landed"]
        att = s[f"{group}_att"]
        absorbed = s[f"opp_{group}_landed"]
        opp_att = s[f"opp_{group}_att"]
        feats[f"{group}_acc"] = rate(landed, att)
        feats[f"{group}_land_per_min"] = per_min(landed, seconds)
        feats[f"{group}_att_per_min"] = per_min(att, seconds)
        feats[f"{group}_absorbed_per_min"] = per_min(absorbed, seconds)
        feats[f"{group}_opp_att_per_min"] = per_min(opp_att, seconds)
        feats[f"{group}_def"] = np.nan if opp_att == 0 else 1 - absorbed / opp_att
    return feats

def elo_expected(a, b):
    return 1.0 / (1.0 + 10 ** ((b - a) / 400.0))

def update_state(state, row, side, won):
    opp = "B" if side == "R" else "R"
    state["total_fights"] += 1
    state["wins"] += int(won)
    state["losses"] += int(not won)
    if won:
        state["current_win_streak"] += 1
        state["current_lose_streak"] = 0
        state["longest_win_streak"] = max(state["longest_win_streak"], state["current_win_streak"])
        method = simplify_win_by(row.get("win_by"))
        state[f"win_by_{method}"] += 1
    else:
        state["current_lose_streak"] += 1
        state["current_win_streak"] = 0
    state["total_rounds_fought"] += 0 if pd.isna(row.get("last_round")) else float(row.get("last_round"))
    state["total_title_bouts"] += int(bool(row.get("title_bout")))
    state["total_fight_time_seconds"] += 0 if pd.isna(row.get("fight_time_seconds")) else float(row.get("fight_time_seconds"))
    for count_stat in COUNT_STATS:
        state["sums"][count_stat] += float(row.get(f"{side}_{count_stat}", 0) or 0)
    state["sums"]["CTRL_seconds"] += float(row.get(f"{side}_CTRL_seconds", 0) or 0)
    for group in STAT_GROUPS:
        state["sums"][f"{group}_landed"] += float(row.get(f"{side}_{group}_landed", 0) or 0)
        state["sums"][f"{group}_att"] += float(row.get(f"{side}_{group}_att", 0) or 0)
        state["sums"][f"opp_{group}_landed"] += float(row.get(f"{opp}_{group}_landed", 0) or 0)
        state["sums"][f"opp_{group}_att"] += float(row.get(f"{opp}_{group}_att", 0) or 0)

states = defaultdict(new_state)
elos = defaultdict(lambda: ELO_START)
rows = []

for _, row in df.sort_values(["date", "fight_id"]).iterrows():
    r_name = row["R_fighter"]
    b_name = row["B_fighter"]
    r_elo = elos[r_name]
    b_elo = elos[b_name]

    out = {
        "fight_id": row["fight_id"],
        "date": row["date"],
        "winner_side": row["winner_side"],
        "title_bout": int(bool(row["title_bout"])),
        "no_of_rounds": row["no_of_rounds"],
        "weight_class": row["weight_class"],
        "R_elo": r_elo,
        "B_elo": b_elo,
        "R_age": row["R_age"],
        "B_age": row["B_age"],
        "R_height": row["R_Height_inches"],
        "B_height": row["B_Height_inches"],
        "R_reach": row["R_Reach_inches"],
        "B_reach": row["B_Reach_inches"],
        "R_weight": row["R_Weight_lbs"],
        "B_weight": row["B_Weight_lbs"],
        "R_stance": row["R_Stance_clean"],
        "B_stance": row["B_Stance_clean"],
    }
    for side, name in [("R", r_name), ("B", b_name)]:
        for key, value in state_to_features(states[name]).items():
            out[f"{side}_{key}"] = value
    rows.append(out)

    r_won = row["winner_side"] == "R"
    update_state(states[r_name], row, "R", r_won)
    update_state(states[b_name], row, "B", not r_won)

    expected_r = elo_expected(r_elo, b_elo)
    score_r = 1.0 if r_won else 0.0
    elos[r_name] = r_elo + ELO_K * (score_r - expected_r)
    elos[b_name] = b_elo + ELO_K * ((1.0 - score_r) - (1.0 - expected_r))

features = pd.DataFrame(rows)
print("Built pre-fight rows:", features.shape)
print(features[["winner_side", "R_total_fights", "B_total_fights", "R_elo", "B_elo"]].head())

Built pre-fight rows: (8520, 166)
  winner_side  R_total_fights  B_total_fights   R_elo   B_elo
0           R               0               0  1500.0  1500.0
1           R               0               0  1500.0  1500.0
2           R               0               0  1500.0  1500.0
3           R               1               0  1516.0  1500.0
4           R               0               0  1500.0  1500.0


## Convert To DIFF Features And Balance Side Labels

In [5]:
def make_diff_features(data):
    out = data[["fight_id", "date", "winner_side", "title_bout", "no_of_rounds", "weight_class"]].copy()
    out["stance_matchup"] = data["R_stance"].astype(str) + "_vs_" + data["B_stance"].astype(str)

    age_center = pd.concat([data["R_age"], data["B_age"]]).median()
    out["DIFF_age_centered_sq"] = (data["R_age"] - age_center) ** 2 - (data["B_age"] - age_center) ** 2

    r_cols = [c for c in data.columns if c.startswith("R_")]
    for r_col in r_cols:
        suffix = r_col[2:]
        b_col = f"B_{suffix}"
        if b_col not in data.columns:
            continue
        if suffix in {"age", "stance"}:
            continue
        r_num = pd.to_numeric(data[r_col], errors="coerce")
        b_num = pd.to_numeric(data[b_col], errors="coerce")
        if r_num.notna().any() or b_num.notna().any():
            out[f"DIFF_{suffix}"] = r_num - b_num
    return out, age_center

model_df, age_center = make_diff_features(features)

def balance_by_side_swap(data, seed=42):
    balanced = data.copy()
    counts = balanced["winner_side"].value_counts()
    majority = counts.idxmax()
    minority = counts.idxmin()
    n_to_flip = int((counts[majority] - counts[minority]) // 2)
    rng = np.random.default_rng(seed)
    idx = balanced.index[balanced["winner_side"] == majority].to_numpy()
    flip_idx = rng.choice(idx, size=n_to_flip, replace=False)

    diff_cols = [c for c in balanced.columns if c.startswith("DIFF_")]
    balanced.loc[flip_idx, diff_cols] = -balanced.loc[flip_idx, diff_cols]
    balanced.loc[flip_idx, "winner_side"] = minority

    def swap_matchup(value):
        parts = str(value).split("_vs_")
        return f"{parts[1]}_vs_{parts[0]}" if len(parts) == 2 else value

    balanced.loc[flip_idx, "stance_matchup"] = balanced.loc[flip_idx, "stance_matchup"].map(swap_matchup)
    balanced["was_side_swapped"] = False
    balanced.loc[flip_idx, "was_side_swapped"] = True
    return balanced

balanced_df = balance_by_side_swap(model_df, seed=RANDOM_STATE)

print("Age center used for DIFF_age_centered_sq:", age_center)
print("Original target distribution:")
print(model_df["winner_side"].value_counts())
print("\nBalanced target distribution:")
print(balanced_df["winner_side"].value_counts())
print("Swapped rows:", int(balanced_df["was_side_swapped"].sum()))
print("Feature columns before one-hot:", balanced_df.shape[1])

Age center used for DIFF_age_centered_sq: 30.27789185489391
Original target distribution:
winner_side
R    5452
B    3068
Name: count, dtype: int64

Balanced target distribution:
winner_side
R    4260
B    4260
Name: count, dtype: int64
Swapped rows: 1192
Feature columns before one-hot: 87


## Evaluate RF And XGBoost With 5 Random 90/10 Splits

In [6]:
def make_xy(data):
    target = data["winner_side"].map({"B": 0, "R": 1}).astype(int)
    drop_cols = ["fight_id", "date", "winner_side", "was_side_swapped"]
    X = data.drop(columns=[c for c in drop_cols if c in data.columns]).copy()
    X = pd.get_dummies(X, columns=["weight_class", "stance_matchup"], dummy_na=True)
    X = X.replace([np.inf, -np.inf], np.nan)
    return X, target

def evaluate_model(name, estimator_factory, X, y):
    rows = []
    last_confusion = None
    last_report = None
    last_importance = None

    for split_num, seed in enumerate(SPLIT_SEEDS, start=1):
        splitter = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
        train_idx, test_idx = next(splitter.split(X, y))
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", estimator_factory(seed)),
        ])
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]
        rows.append({
            "model": name,
            "split": split_num,
            "seed": seed,
            "accuracy": accuracy_score(y_test, pred),
            "roc_auc": roc_auc_score(y_test, proba),
            "train_rows": len(train_idx),
            "test_rows": len(test_idx),
        })
        last_confusion = confusion_matrix(y_test, pred)
        last_report = classification_report(y_test, pred, target_names=["B", "R"])
        fitted = model.named_steps["model"]
        if hasattr(fitted, "feature_importances_"):
            last_importance = pd.Series(fitted.feature_importances_, index=X.columns).sort_values(ascending=False)

    splits = pd.DataFrame(rows)
    summary = pd.DataFrame([{
        "model": name,
        "n_features": X.shape[1],
        "mean_accuracy": splits["accuracy"].mean(),
        "std_accuracy": splits["accuracy"].std(),
        "mean_auc": splits["roc_auc"].mean(),
        "std_auc": splits["roc_auc"].std(),
    }])
    return summary, splits, last_confusion, last_report, last_importance

X, y = make_xy(balanced_df)
print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts(normalize=True))

rf_factory = lambda seed: RandomForestClassifier(random_state=seed, **RF_PARAMS)
xgb_factory = lambda seed: XGBClassifier(random_state=seed, **XGB_PARAMS)

rf_summary, rf_splits, rf_confusion, rf_report, rf_importance = evaluate_model("Random Forest", rf_factory, X, y)
xgb_summary, xgb_splits, xgb_confusion, xgb_report, xgb_importance = evaluate_model("XGBoost", xgb_factory, X, y)

summary = pd.concat([rf_summary, xgb_summary], ignore_index=True).sort_values("mean_accuracy", ascending=False)
splits = pd.concat([rf_splits, xgb_splits], ignore_index=True)

display(summary)
display(splits.pivot(index="split", columns="model", values="accuracy"))

X shape: (8520, 112)
y distribution:
winner_side
1    0.5
0    0.5
Name: proportion, dtype: float64


,model,n_features,mean_accuracy,std_accuracy,mean_auc,std_auc
1,XGBoost,112,0.625117,0.009189,0.670397,0.011551
0,Random Forest,112,0.620423,0.011900,0.665932,0.013704


model,Random Forest,XGBoost
split,,
1,0.612676,0.625587
2,0.620892,0.629108
3,0.626761,0.630282
4,0.605634,0.609155
5,0.636150,0.631455


## Last Split Diagnostics

In [7]:
print("Random Forest last split confusion matrix:")
print(rf_confusion)
print(rf_report)
display(rf_importance.head(30).to_frame("importance"))

print("\nXGBoost last split confusion matrix:")
print(xgb_confusion)
print(xgb_report)
display(xgb_importance.head(30).to_frame("importance"))

Random Forest last split confusion matrix:
[[277 149]
 [161 265]]
              precision    recall  f1-score   support

           B       0.63      0.65      0.64       426
           R       0.64      0.62      0.63       426

    accuracy                           0.64       852
   macro avg       0.64      0.64      0.64       852
weighted avg       0.64      0.64      0.64       852



,importance
weight_class_Unknown,0.043817
DIFF_elo,0.043539
DIFF_age_centered_sq,0.026937
DIFF_total_fight_time_seconds,0.020438
DIFF_win_pct,0.017880
DIFF_TOTAL_STR_absorbed_per_min,0.017300
DIFF_TD_land_per_min,0.016054
DIFF_HEAD_def,0.015622
DIFF_total_rounds_fought,0.015446
DIFF_reach,0.014867



XGBoost last split confusion matrix:
[[271 155]
 [159 267]]
              precision    recall  f1-score   support

           B       0.63      0.64      0.63       426
           R       0.63      0.63      0.63       426

    accuracy                           0.63       852
   macro avg       0.63      0.63      0.63       852
weighted avg       0.63      0.63      0.63       852



,importance
weight_class_Unknown,0.078228
DIFF_elo,0.024722
DIFF_TOTAL_STR_absorbed_per_min,0.019420
DIFF_current_win_streak,0.018416
DIFF_losses,0.017612
title_bout,0.017527
DIFF_win_pct,0.015186
weight_class_Bantamweight,0.014839
DIFF_TD_land_per_min,0.013995
DIFF_GROUND_absorbed_per_min,0.013832
